# Project 1 Learning and Self-Assessment Workbook

Use this notebook **after the Calgary Spatial ETL pipeline works end to end**. It is separate from the walkthrough because its purpose is retrieval practice, not instruction.

The learning sequence is:

1. Recall the workflow without notes.
2. Predict what code will do before running it.
3. Complete fading templates.
4. Build functions from requirements only.
5. Modify and debug unfamiliar cases.
6. Explain your reasoning as if teaching someone else.
7. Review weak areas using spaced repetition.

Do not try to memorize the entire project. Aim to remember the workflow, choose appropriate Python structures, explain important decisions, and verify behavior independently. Looking up exact library syntax after first attempting recall is normal professional practice.

## Tutor Rules

For the most useful assessment:

- Close the walkthrough and line-by-line guides before starting.
- Write an answer or prediction before running a check.
- Do not open the source until an exercise tells you to compare.
- Record errors instead of erasing all evidence of them.
- Use the answer keys only after making a genuine attempt.
- Give yourself partial credit when your reasoning is sound but syntax is imperfect.
- Bring written responses and failed checks back to Copilot for tutor feedback.

A good result is not a perfect first attempt. A good result identifies exactly what you understand and what needs another retrieval session.

## Session Record

Before each attempt, enter the date and whether this is your first attempt, 1-day review, 3-day review, 1-week review, or 2-week review.

In [ ]:
from datetime import date

assessment_date = date.today().isoformat()
review_interval = "first attempt"  # Change for later reviews.
confidence_before = 0  # Enter a number from 0 to 5 before starting.

print(f"Date: {assessment_date}")
print(f"Review: {review_interval}")
print(f"Starting confidence: {confidence_before}/5")

## Readiness Gate: Confirm the Project Works First

This notebook should assess knowledge of working behavior. It should not be used to compensate for an unfinished or failing pipeline.

The next cell checks the project root, raw and processed files, transform log, CRS, and geometry validity. If it reports a failure, return to the walkthrough and repair the project before continuing.

In [ ]:
from pathlib import Path
import os
import sys
import geopandas as gpd

def find_project_root(start: Path) -> Path:
    markers = ["environment.yml", "docker-compose.yml", "src"]
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError("Could not locate the project root.")

project_root = find_project_root(Path.cwd())
os.chdir(project_root)

datasets = ["communities", "roads", "transit_stops", "land_use_districts"]
problems = []

for dataset in datasets:
    raw_path = Path(f"data/raw/{dataset}.geojson")
    processed_path = Path(f"data/processed/{dataset}.geojson")
    if not raw_path.exists():
        problems.append(f"Missing raw file: {raw_path}")
    if not processed_path.exists():
        problems.append(f"Missing processed file: {processed_path}")
        continue
    layer = gpd.read_file(processed_path)
    if str(layer.crs) != "EPSG:3347":
        problems.append(f"{dataset}: expected EPSG:3347, got {layer.crs}")
    if int(layer.geometry.isna().sum()) > 0:
        problems.append(f"{dataset}: contains null geometries")
    if int((~layer.is_valid).sum()) > 0:
        problems.append(f"{dataset}: contains invalid geometries")

if not Path("outputs/logs/transform_log.csv").exists():
    problems.append("Missing transform log")

print(f"Project root: {project_root}")
print(f"Python kernel: {sys.executable}")
if problems:
    print("NOT READY:")
    for problem in problems:
        print(f"- {problem}")
else:
    print("READY: current Extract and Transform outputs passed the workbook gate.")

## Scoring System

The workbook uses two kinds of assessment:

- **Executable checks:** code receives credit only when its tests pass.
- **Tutor or self review:** written explanations use a rubric because wording can vary while still being correct.

For each written answer, award:

- `0`: blank, guessed, or fundamentally incorrect
- `1`: partly correct but missing an important distinction
- `2`: correct and explained in your own words with a relevant example

Do not award full credit merely because your answer resembles the key after you read it. Score the answer you wrote before revealing the key.

In [ ]:
scores = {}

def record_score(exercise: str, earned: int, possible: int) -> None:
    if not 0 <= earned <= possible:
        raise ValueError("Score must be between zero and the possible points.")
    scores[exercise] = (earned, possible)
    print(f"Recorded {exercise}: {earned}/{possible}")

def run_check(exercise: str, check_function, possible: int = 2) -> None:
    try:
        check_function()
    except Exception as error:
        scores[exercise] = (0, possible)
        print(f"NOT YET: {exercise} - {type(error).__name__}: {error}")
    else:
        scores[exercise] = (possible, possible)
        print(f"PASS: {exercise} - {possible}/{possible}")

# Stage 1: Recall the Workflow

Without opening project files, answer in your own words:

1. What is the purpose of Extract, Transform, QA, and Load?
2. What enters and leaves the Transform stage?
3. Why does the project use one target CRS?
4. Why are configuration and processing logic kept in different modules?
5. Name at least four things that can fail in a spatial ETL pipeline.

Write concise answers below. Do not aim for memorized wording.

In [ ]:
workflow_answers = {
    "etl_stages": "",
    "transform_inputs_outputs": "",
    "common_crs": "",
    "config_separation": "",
    "failure_examples": "",
}

for question, answer in workflow_answers.items():
    print(f"{question}: {answer or '[not answered]'}")

<details>
<summary>Stage 1 review guide - open only after answering</summary>

A strong answer should include these ideas:

- Extract retrieves source data and records provenance.
- Transform standardizes schema, IDs, geometry, and CRS and writes processed data.
- QA measures whether outputs meet explicit quality expectations.
- Load writes validated data into PostGIS and creates useful database structures such as spatial indexes.
- Transform receives raw geospatial files and configuration and produces standardized processed files plus log records.
- A common CRS allows meaningful spatial comparison and operations; projected EPSG:3347 also gives suitable Canadian metric coordinates.
- Configuration can change without rewriting reusable processing logic.
- Failures can include unavailable URLs, missing files or fields, wrong CRS, invalid geometry, database connection problems, permission errors, and schema changes.

Award 0-2 points for each of the five answers, then record the total out of 10.
</details>

In [ ]:
record_score("stage_1_workflow_recall", earned=0, possible=10)  # Replace 0 after reviewing.

# Stage 2: Predict Before Running

For each expression, write your predicted result first. Then run the following cell and compare.

1. What does `" Road---CLASS  ".strip().lower()` return?
2. What does `re.sub(r"[^a-z0-9]+", "_", "road---class")` return?
3. What does `re.sub(r"_+", "_", "__road___class__").strip("_")` return?
4. What does `Path("data/raw/roads.geojson").parent` return?
5. Is `None and None in ["id"]` truthy or falsy? Why does the second expression not need to run?

In [ ]:
prediction_answers = {
    "strip_lower": "",
    "non_alphanumeric_regex": "",
    "underscore_regex": "",
    "path_parent": "",
    "short_circuit": "",
}
prediction_answers

In [ ]:
import re
from pathlib import Path

actual_results = {
    "strip_lower": " Road---CLASS  ".strip().lower(),
    "non_alphanumeric_regex": re.sub(r"[^a-z0-9]+", "_", "road---class"),
    "underscore_regex": re.sub(r"_+", "_", "__road___class__").strip("_"),
    "path_parent": str(Path("data/raw/roads.geojson").parent),
    "short_circuit": bool(None and None in ["id"]),
}

for name, actual in actual_results.items():
    print(f"{name}: predicted={prediction_answers[name]!r}, actual={actual!r}")

Award 1 point for each correct prediction made before running the results cell. For short-circuit evaluation, the prediction must also explain that `and` stops after a falsy first operand.

In [ ]:
record_score("stage_2_prediction", earned=0, possible=5)  # Replace after comparing.

# Stage 3: Fading Template

Complete the function without looking at `src/transform.py`.

Requirements:

- Remove surrounding whitespace.
- Convert letters to lowercase.
- Replace consecutive non-alphanumeric characters with one underscore.
- Collapse repeated underscores.
- Remove underscores from both ends.
- Return the normalized string.

Replace `pass` with your implementation, then run the check cell.

In [ ]:
def student_normalize_col_name(name: str) -> str:
    pass

In [ ]:
def check_normalize_col_name() -> None:
    assert student_normalize_col_name(" Road Name " ) == "road_name"
    assert student_normalize_col_name("ROAD---TYPE") == "road_type"
    assert student_normalize_col_name("__Stop   ID__") == "stop_id"
    assert student_normalize_col_name("class_code") == "class_code"

run_check("stage_3_normalize_function", check_normalize_col_name, possible=4)

## Stage 3B: Expand a List Comprehension

Write `student_normalize_columns` using a regular `for` loop rather than a list comprehension. It must copy the GeoDataFrame, build a new list of normalized column names, assign the list to `.columns`, and return the copy.

In [ ]:
def student_normalize_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    pass

In [ ]:
from shapely.geometry import Point

def check_normalize_columns() -> None:
    original = gpd.GeoDataFrame(
        {" Road Name " : ["Main"], "Stop---ID": [101]},
        geometry=[Point(-114.07, 51.05)],
        crs="EPSG:4326",
    )
    result = student_normalize_columns(original)
    assert list(result.columns) == ["road_name", "stop_id", "geometry"]
    assert list(original.columns) == [" Road Name ", "Stop---ID", "geometry"]
    assert result is not original

run_check("stage_3_normalize_columns", check_normalize_columns, possible=4)

# Stage 4: Build From Requirements Only

Implement a field-selection function without consulting the project source.

Requirements:

- Accept a GeoDataFrame and a list of expected field names.
- Do not mutate the caller's GeoDataFrame.
- For each absent expected field, create a column filled with `None`.
- Record each absent field name in a list.
- Return only expected fields in their requested order, followed by geometry.
- Return both the resulting GeoDataFrame and missing-fields list.

In [ ]:
def student_keep_required_fields(
    gdf: gpd.GeoDataFrame, keep_fields: list[str]
) -> tuple[gpd.GeoDataFrame, list[str]]:
    pass

In [ ]:
def check_keep_required_fields() -> None:
    original = gpd.GeoDataFrame(
        {"name": ["A"], "sector": ["North"], "extra": [99]},
        geometry=[Point(-114.07, 51.05)],
        crs="EPSG:4326",
    )
    result, missing = student_keep_required_fields(
        original, ["name", "sector", "population"]
    )
    assert list(result.columns) == ["name", "sector", "population", "geometry"]
    assert missing == ["population"]
    assert result["population"].isna().all()
    assert "population" not in original.columns
    assert "extra" in original.columns and "extra" not in result.columns

run_check("stage_4_field_selection", check_keep_required_fields, possible=5)

## Stage 4B: ID Conversion and Short-Circuit Logic

Implement a function that copies a GeoDataFrame and converts a configured ID column to pandas `string` dtype only when the configured ID is neither `None` nor absent.

In [ ]:
def student_cast_id_to_string(
    gdf: gpd.GeoDataFrame, id_field: str | None
) -> gpd.GeoDataFrame:
    pass

In [ ]:
def check_cast_id_to_string() -> None:
    original = gpd.GeoDataFrame(
        {"feature_id": [101, 102]},
        geometry=[Point(0, 0), Point(1, 1)],
        crs="EPSG:4326",
    )
    converted = student_cast_id_to_string(original, "feature_id")
    unchanged = student_cast_id_to_string(original, None)
    absent = student_cast_id_to_string(original, "not_a_column")
    assert str(converted["feature_id"].dtype) == "string"
    assert str(original["feature_id"].dtype) != "string"
    assert unchanged.equals(original)
    assert absent.equals(original)

run_check("stage_4_id_conversion", check_cast_id_to_string, possible=4)

# Stage 5: Modify the Requirement

A developer must adapt code, not only reproduce it. Build an improved normalizer with this additional requirement:

- If normalization produces an empty string, return `"unnamed_column"`.

Do not change your earlier function. Implement the variation separately.

In [ ]:
def student_normalize_with_fallback(name: str) -> str:
    pass

In [ ]:
def check_normalize_with_fallback() -> None:
    assert student_normalize_with_fallback("Road Name") == "road_name"
    assert student_normalize_with_fallback("---") == "unnamed_column"
    assert student_normalize_with_fallback("   " ) == "unnamed_column"

run_check("stage_5_modified_normalizer", check_normalize_with_fallback, possible=3)

## Stage 5B: Diagnose Before Fixing

The following function is supposed to convert a raw output path into a processed output path, but it has a bug:

```python
def broken_processed_output_path(raw_output_path: str) -> Path:
    raw_output_path.replace("data/raw/", "data/processed/")
    return Path(raw_output_path)
```

Before editing anything, answer:

1. What output will it return for `data/raw/roads.geojson`?
2. Why does `.replace()` not change `raw_output_path`?
3. Is this primarily a namespace problem, a mutation/rebinding problem, or a path-library problem?
4. Write the smallest correction.

In [ ]:
debug_answers = {
    "predicted_output": "",
    "why_replace_did_not_change_it": "",
    "problem_category": "",
    "smallest_correction": "",
}
debug_answers

<details>
<summary>Stage 5B answer key - open after answering</summary>

1. It returns `Path("data/raw/roads.geojson")`.
2. Python strings are immutable. `.replace()` returns a new string; it does not mutate the original string.
3. It is a return-value/rebinding mistake involving an immutable object, not a namespace or `Path` failure.
4. A minimal correction is `return Path(raw_output_path.replace("data/raw/", "data/processed/"))`.

Award 0-2 points for each answer.
</details>

In [ ]:
record_score("stage_5_debug_reasoning", earned=0, possible=8)  # Replace after reviewing.

# Stage 6: Teach-Back

Answer each prompt as if explaining it to a new GIS student. Use your own example rather than copying project wording.

1. Explain a function's local namespace versus the module namespace.
2. Explain why `gdf = gdf.copy()` protects the caller from later GeoDataFrame mutations.
3. Explain rebinding a name versus mutating an object.
4. Explain `set_crs()` versus `to_crs()` and why confusing them is dangerous.
5. Explain how a Boolean mask filters a GeoDataFrame.
6. Explain tuple packing and unpacking using `gdf, missing = ...`.
7. Explain why `land_use_districts` has `None` in `ID_FIELDS`.
8. Explain why silently creating missing columns is useful but risky.

In [ ]:
teach_back_answers = {
    "namespaces": "",
    "gdf_copy": "",
    "rebinding_vs_mutation": "",
    "set_crs_vs_to_crs": "",
    "boolean_mask": "",
    "tuple_unpacking": "",
    "land_use_id_none": "",
    "missing_column_tradeoff": "",
}

for topic, answer in teach_back_answers.items():
    print(f"{topic}: {answer or '[not answered]'}")

## Teach-Back Rubric

Award up to 2 points per response:

- `0`: missing or incorrect
- `1`: core idea is present but an important distinction or example is missing
- `2`: accurate, in your own words, and supported with a relevant example or consequence

For tutor grading, send the `teach_back_answers` output to Copilot and ask: **Grade these against the Stage 6 rubric, identify misconceptions, and ask me one follow-up question for each weak topic without immediately giving me the answer.**

In [ ]:
record_score("stage_6_teach_back", earned=0, possible=16)  # Replace after self or tutor review.

# Stage 7: Independent Mini-Task

Use a new or unfamiliar GeoJSON file. You may consult official documentation after first writing your plan, but do not follow the project source line by line.

Your task:

1. Read the file.
2. Inspect row count, columns, geometry types, and CRS.
3. Normalize its column names.
4. Select a small useful schema.
5. Check null, empty, and invalid geometries.
6. Reproject it to an appropriate projected CRS.
7. Save it to a new output path.
8. Reopen the saved file and verify the result.

Before coding, write your plan and expected verification evidence below.

In [ ]:
independent_task_plan = ""
expected_evidence = ""

print(independent_task_plan or "[write your plan]")
print(expected_evidence or "[write expected evidence]")

In [ ]:
# Implement the independent mini-task here or in a separate practice file.
# Keep this cell blank until you have written the plan above.

Score the independent task out of 8, one point for each completed step. Only award a point when you produced evidence, such as printed metadata, an assertion, or a successfully reopened output file.

In [ ]:
record_score("stage_7_independent_task", earned=0, possible=8)  # Replace after completion.

# Results and Tutor Review

Run the summary cell after recording every score. Suggested interpretation:

- **85-100%:** strong working understanding; move to a variation or independent task
- **70-84%:** usable understanding with specific review areas
- **50-69%:** repeat weak stages after one day
- **Below 50%:** return to the explanations, then retry with fading templates

The percentage is diagnostic, not a credential. Improvement across spaced attempts matters more than one score.

In [ ]:
earned_total = sum(earned for earned, possible in scores.values())
possible_total = sum(possible for earned, possible in scores.values())
percentage = (earned_total / possible_total * 100) if possible_total else 0

print("Assessment summary")
print("------------------")
for exercise, (earned, possible) in scores.items():
    print(f"{exercise}: {earned}/{possible}")
print(f"Total: {earned_total}/{possible_total} ({percentage:.1f}%)")

weak_areas = [
    exercise
    for exercise, (earned, possible) in scores.items()
    if possible and earned / possible < 0.7
]
print(f"Review next: {weak_areas if weak_areas else 'No stage below 70%'}")

## Tutor Review Prompt

After completing an attempt, ask Copilot to review the notebook with this instruction:

> Act as my GIS/Python tutor. Review my answers, code, failed checks, and score summary in `learning/assessments/project1_learning_assessment.ipynb`. Do not rewrite everything immediately. First identify what I understand, then identify misconceptions, ask targeted follow-up questions, and give one smaller practice exercise for each weak area. Grade written answers using the notebook rubric and preserve my original attempts.

This keeps the feedback diagnostic and prevents the tutor from replacing your thinking with finished answers.

# Spaced Review Schedule

Repeat only the weak stages on this schedule:

- Same day: correct misunderstandings and explain the correction.
- 1 day later: retry without notes.
- 3 days later: retry with a changed example.
- 1 week later: complete requirements-only exercises.
- 2 weeks later: complete the independent mini-task with a different dataset.

Do not repeatedly rerun passing cells just to remember their output. Change field names, paths, missing columns, geometry examples, or CRS so each review requires fresh reasoning.

# QA and Load Review Extensions

The project now has working QA and Load modules plus an end-to-end runner. Extend later assessment attempts with these verified topics:

- Predict which schema, ID, CRS, and geometry defects should block loading.
- Interpret `QAResult` metrics and explain why a failed gate prevents database writes.
- Compare `append`, `replace`, and upsert behavior; explain why this project uses repeatable replacement.
- Explain how one transaction protects a multi-layer load from partial completion.
- Verify source and destination row counts, SRID `3347`, and GIST geometry indexes.
- Force a controlled failure in an isolated schema and prove that rollback removed partial writes.
- Run `python -m src.main --skip-extract` and reconcile Transform, QA, and Load evidence.

Use the QA and Load practice notebooks before adding scored assessment cells. Preserve first attempts and grade explanations with the existing 0-2 rubric.